# SG Job Data — Exploratory Data Analysis

This notebook explores the raw `SGJobData.csv` to understand data quality and inform the cleaning pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../SGJobData.csv", low_memory=False)
print(f"Shape: {df.shape}")
df.head(3)

## 1. Shape & Types

In [ ]:
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Missing Values

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"count": missing, "pct": missing_pct})[missing > 0]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, ax=ax)
ax.set_title("Missing Values Heatmap")
plt.tight_layout()
plt.show()

## 3. Duplicates

In [ ]:
dup_count = df["metadata_jobPostId"].duplicated().sum()
print(f"Duplicate job post IDs: {dup_count} ({dup_count / len(df) * 100:.2f}%)")

## 4. Categories Column

In [ ]:
def parse_primary_category(cat_str):
    try:
        cats = json.loads(cat_str)
        return cats[0]["category"] if cats else None
    except (json.JSONDecodeError, TypeError, KeyError, IndexError):
        return None

cats = df["categories"].apply(parse_primary_category)
cat_counts = cats.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 8))
cat_counts.plot.barh(ax=ax)
ax.set_title("Top 20 Primary Categories")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

print(f"\nNull categories after parsing: {cats.isnull().sum()}")
print(f"Unique categories: {cats.nunique()}")

## 5. Salary Analysis

In [ ]:
print("Salary type distribution:")
print(df["salary_type"].value_counts())

print(f"\nRows where both salary_min and salary_max are 0: {((df['salary_minimum'] == 0) & (df['salary_maximum'] == 0)).sum()}")
print(f"Rows where salary_min > salary_max: {(df['salary_minimum'] > df['salary_maximum']).sum()}")

In [ ]:
valid_sal = df[(df["salary_minimum"] > 0) | (df["salary_maximum"] > 0)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(["salary_minimum", "salary_maximum", "average_salary"]):
    axes[i].hist(valid_sal[col].dropna(), bins=50, edgecolor="black")
    axes[i].set_title(col)
    axes[i].set_xlabel("SGD")
plt.tight_layout()
plt.show()

print("\nSalary stats (non-zero):")
valid_sal[["salary_minimum", "salary_maximum", "average_salary"]].describe()

In [ ]:
p99 = valid_sal["average_salary"].quantile(0.99)
print(f"99th percentile of average_salary: {p99:.0f}")
print(f"Max average_salary: {valid_sal['average_salary'].max():.0f}")
print(f"Rows above 99th pct: {(valid_sal['average_salary'] > p99).sum()}")

## 6. Date Fields

In [ ]:
for col in ["metadata_newPostingDate", "metadata_expiryDate", "metadata_originalPostingDate"]:
    parsed = pd.to_datetime(df[col], errors="coerce")
    invalid = parsed.isnull().sum() - df[col].isnull().sum()
    print(f"{col}: range [{parsed.min()} — {parsed.max()}], invalid parses: {invalid}")

## 7. Text Fields

In [ ]:
title_lens = df["title"].str.len()
fig, ax = plt.subplots(figsize=(10, 4))
title_lens.hist(bins=50, ax=ax, edgecolor="black")
ax.set_title("Title Length Distribution")
ax.set_xlabel("Characters")
plt.tight_layout()
plt.show()

print(f"Unique companies: {df['postedCompany_name'].nunique()}")
print(f"\nSample company names (first 10):")
print(df["postedCompany_name"].value_counts().head(10))

## Summary of Cleaning Decisions

1. **Drop duplicates** by `metadata_jobPostId`
2. **Parse categories** JSON → extract primary category (first item's `category` field)
3. **Normalize salary**: drop rows where salary_min = 0 AND salary_max = 0; cap at 99th percentile
4. **Parse dates**: drop rows with invalid `metadata_newPostingDate`
5. **Fill missing**: `minimumYearsExperience` → 0, `numberOfVacancies` → 1
6. **Standardize company names**: strip whitespace, title-case
7. **Rename columns** to clean snake_case matching DB schema